# DSNet — OASIS-1 FreeSurfer 二分类（NonDemented vs Demented）

## 架构（与 Kaggle 版完全相同，仅输出层改为 2 类）

```
Input (224×224×3, 轴向 MRI 切片 RGB)
        │
┌───────┴──────────────────────────────────────┐
│  DenseNet-121 CNN Extractor (预训练，截断)      │
│  conv0→norm0→relu0→pool0                     │
│  → denseblock1 → transition1  [B,128,28,28]  │
│  → denseblock2 → transition2  [B,256,14,14]  │
│  → denseblock3                [B,1024,14,14] │
└───────────────────────────────────────────────┘
        │
┌───────┴──────────────────────────────────────┐
│  Feature Projection  (新模块，可训练)           │
│  BN2d + Conv2d(1024→384, 1×1) + GELU        │
│  Permute(0,2,3,1)   [B,14,14,384]  NHWC     │
└───────────────────────────────────────────────┘
        │
┌───────┴──────────────────────────────────────┐
│  Swin Stage 3 (预训练, 6块, W-MSA/SW-MSA)     │
│  → PatchMerging    [B, 7, 7, 768]            │
│  → Swin Stage 4 (预训练, 2块)  ← Grad-CAM    │
└───────────────────────────────────────────────┘
        │
┌───────┴──────────────────────────────────────┐
│  Classification Head                          │
│  LayerNorm → AvgPool → Dropout(0.3) → FC(2) │  ← 二分类
└───────────────────────────────────────────────┘
```

## 数据流
OASIS-1 FreeSurfer `brainmask.mgz` (3D, 256³) → 预处理为轴向 2D PNG 切片 → 224×224 RGB → DSNet

| 标签 | CDR | 受试者数（约） |
|------|-----|---------------|
| NonDemented | 0 | ~300 |
| Demented | 0.5 / 1 / 2 | ~116 |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nibabel', 'grad-cam', 'openpyxl', '-q'])
print('Setup complete.')

In [ ]:
import hashlib, os, time
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from PIL import Image
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
)
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision.models import DenseNet121_Weights, Swin_T_Weights
from tqdm.notebook import tqdm

# ── Configuration ─────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive'
OASIS_DIR    = f'{DRIVE_ROOT}/OASIS1_FreeSurfer'
XLSX_PATH    = f'{OASIS_DIR}/oasis_cross-sectional-5708aa0a98d82080.xlsx'
SLICES_DIR   = f'{DRIVE_ROOT}/OASIS1_slices_triview'
OUTPUT_DIR   = f'{DRIVE_ROOT}/alzheimer_dsnet_oasis_outputs'
CHECKPOINT_DIR = f'{OUTPUT_DIR}/checkpoints'
GRADCAM_DIR  = f'{OUTPUT_DIR}/gradcam'
RESULTS_DIR  = f'{OUTPUT_DIR}/results'

for d in [CHECKPOINT_DIR, GRADCAM_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

CLASS_ORDER   = ['NonDemented', 'Demented']
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

VIEW_CONFIGS = {
    'axial':    {'dim': 2, 'range': (0.27, 0.47)},
    'coronal':  {'dim': 1, 'range': (0.35, 0.57)},
    'sagittal': {'dim': 0, 'range': (0.31, 0.68)},
}
N_SLICES = 40   # 原 20，每名受试者采样两倍切片（5800 训练图/视角 vs 原 2900）

BATCH_SIZE    = 32
NUM_WORKERS   = 4
USE_AMP       = True

# 小数据集（~153 训练受试者）需要更多轮次 + 更大耐心防止噪声触发过早停止
PHASE1_EPOCHS = 15
PHASE2_EPOCHS = 20
PATIENCE      = 7
MIN_DELTA     = 0.001

PHASE1_LR      = 1e-4

# Phase 2 LR（CNN/Swin 极低，保持不变）
PHASE2_LR_CNN  = 1e-6
PHASE2_LR_SWIN = 5e-7

# 视角专用 Phase 2 proj 学习率
VIEW_PHASE2_LR_PROJ = {
    'axial':    1e-5,
    'coronal':  1e-5,
    'sagittal': 5e-5,
}

WEIGHT_DECAY   = 1e-4
N_GRADCAM      = 5

print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}')
    print(f'VRAM   : {props.total_memory / 1e9:.1f} GB')
print(f'Views  : {list(VIEW_CONFIGS.keys())}')
print(f'N_SLICES={N_SLICES}  PHASE1_EPOCHS={PHASE1_EPOCHS}  PATIENCE={PATIENCE}')
print(f'Phase 2 proj LR (per-view): {VIEW_PHASE2_LR_PROJ}')

In [ ]:
# ── 读取 OASIS-1 CDR 标签，构建受试者列表（全数据集 ~416 人）──────
# 包含：CDR=0 老年健康对照（60-94岁）+ CDR=NaN 年轻健康对照（18-35岁）+ CDR>0 痴呆患者
#
# 年龄混杂说明（论文局限性声明）：
# OASIS-1 中 CDR=NaN 的年轻健康对照与老年 CDR=0 均标记为 NonDemented。
# 这与原始 OASIS-1 研究设计一致（Marcus et al., 2007）。
# 年龄匹配敏感性分析（仅老年受试者 N=219）的 AUC=0.65，
# 相比之下全数据集 AUC=0.80，验证了年龄混杂对模型性能的贡献。

df_all = pd.read_excel(XLSX_PATH)
df_all['subject_id'] = df_all['ID'].str.strip()

# Excel ID → CDR 值映射（保留 NaN）
cdr_map = dict(zip(df_all['subject_id'], df_all['CDR']))

# 递归搜索所有 brainmask.mgz（数据在 disc1...disc11 子目录下）
print('正在扫描 brainmask.mgz 文件（首次运行约需 30 秒）...')
mgz_map = {p.parts[-3]: str(p)
           for p in Path(OASIS_DIR).glob('**/mri/brainmask.mgz')}
assert mgz_map, f'未找到任何 brainmask.mgz！请检查 OASIS_DIR={OASIS_DIR}'
print(f'共找到 {len(mgz_map)} 个 MGZ 文件')

# 只保留 MR1（过滤 MR2/MR3，避免同一受试者多次扫描跨 split 造成数据泄漏）
mgz_mr1 = {sid: path for sid, path in mgz_map.items() if sid.endswith('_MR1')}
print(f'过滤后 MR1 受试者：{len(mgz_mr1)} 个')

valid = []
for sid, mgz_path in mgz_mr1.items():
    cdr = cdr_map.get(sid, float('nan'))
    if pd.isna(cdr):
        # 年轻健康对照（18-35岁，CDR=NaN）→ NonDemented
        # 与 OASIS-1 原始研究设计一致；年龄混杂在论文局限性中声明
        label   = 0
        cdr_val = -1.0
    else:
        label   = int(cdr > 0)   # CDR=0 → NonDemented；CDR>0 → Demented
        cdr_val = float(cdr)
    valid.append({'subject_id': sid, 'label': label, 'mgz': mgz_path, 'cdr': cdr_val})

subjects_df = pd.DataFrame(valid)
cnt = subjects_df['label'].value_counts()
young = (subjects_df['cdr'] == -1.0).sum()
print(f'\n有效受试者 : {len(subjects_df)}')
print(f'  NonDemented 合计                   : {cnt.get(0, 0)}')
print(f'    其中 CDR=0（老年健康，60-94岁）  : {cnt.get(0, 0) - young}')
print(f'    其中 CDR=NaN（年轻对照，18-35岁）: {young}')
print(f'  Demented    (CDR>0)                : {cnt.get(1, 0)}')
print(f'\nCDR 分布（-1=年轻健康对照）:')
print(subjects_df['cdr'].value_counts().sort_index())

In [ ]:
# ── 三视角 MGZ → PNG 预处理（年龄匹配版，仅老年受试者）────────────
# 目录结构: SLICES_DIR/{axial|coronal|sagittal}/{train|val|test}/{NonDemented|Demented}/OAS1_XXXX_MR1_z{i:02d}.png
#
# 重要：先删除旧切片目录（旧版包含年轻健康对照），再重新生成

import shutil
print(f'清理旧切片目录: {SLICES_DIR}')
shutil.rmtree(SLICES_DIR, ignore_errors=True)
print('完成，开始生成年龄匹配版切片...\n')


def subject_split(subject_id: str) -> str:
    h = int(hashlib.md5(subject_id.encode()).hexdigest(), 16) % 100
    return 'train' if h < 70 else ('val' if h < 85 else 'test')

def get_slice(vol: np.ndarray, view: str, idx: int) -> np.ndarray:
    if view == 'axial':    return vol[:, :, idx]
    elif view == 'coronal': return vol[:, idx, :]
    else:                   return vol[idx, :, :]   # sagittal

# 创建所有目录
for view in VIEW_CONFIGS:
    for split in ['train', 'val', 'test']:
        for cls in CLASS_ORDER:
            os.makedirs(f'{SLICES_DIR}/{view}/{split}/{cls}', exist_ok=True)

skipped, processed = 0, 0

for _, row in tqdm(subjects_df.iterrows(), total=len(subjects_df), desc='三视角 MGZ→PNG'):
    sid   = row['subject_id']
    cls   = CLASS_ORDER[row['label']]
    split = subject_split(sid)

    # 检查是否三个视角都已处理完
    all_done = all(
        len(list(Path(f'{SLICES_DIR}/{view}/{split}/{cls}').glob(f'{sid}_z*.png'))) >= N_SLICES
        for view in VIEW_CONFIGS
    )
    if all_done:
        processed += 1
        continue

    vol = nib.load(row['mgz']).get_fdata()   # (256, 256, 256)

    for view, cfg in VIEW_CONFIGS.items():
        dim   = cfg['dim']
        lo, hi = cfg['range']
        n     = vol.shape[dim]
        indices = np.linspace(int(n * lo), int(n * hi) - 1, N_SLICES, dtype=int)

        for i, idx in enumerate(indices):
            out_path = f'{SLICES_DIR}/{view}/{split}/{cls}/{sid}_z{i:02d}.png'
            if os.path.exists(out_path):
                continue
            s = get_slice(vol, view, idx)
            if s.max() < 1e-3:
                skipped += 1
                continue
            s = (s - s.min()) / (s.max() - s.min() + 1e-8)
            s = (s * 255).astype(np.uint8)
            Image.fromarray(s).save(out_path)

    processed += 1

print(f'预处理完成: {processed} 受试者，跳过空白切片 {skipped} 张')
print('\n各视角统计:')
for view in VIEW_CONFIGS:
    for split in ['train', 'val', 'test']:
        nd = len(list(Path(f'{SLICES_DIR}/{view}/{split}/NonDemented').glob('*.png')))
        d  = len(list(Path(f'{SLICES_DIR}/{view}/{split}/Demented').glob('*.png')))
        print(f'  {view:10s} {split:6s}: NonDemented={nd:4d}, Demented={d:4d}')

In [ ]:
# ── Dataset & DataLoader（三视角版，强化数据增强 v2）──────────────────────
# 针对小数据集（~153训练受试者 × 40切片 = 5800张）的增强策略：
#   v1 (已有): Resize(256)→RandomCrop(224), Affine(±15°,±5%,0.9-1.1),
#              HFlip(axial/coronal), ColorJitter(0.3), RandomErasing(0.2)
#   v2 (新增): RandomVerticalFlip(axial专用), RandomPerspective, GaussianBlur

def build_transforms(split: str, view: str = 'axial') -> transforms.Compose:
    norm = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if split == 'train':
        aug = [
            transforms.Resize(256),
            transforms.RandomCrop(224),
            transforms.RandomAffine(
                degrees=15,
                translate=(0.05, 0.05),
                scale=(0.9, 1.1),
            ),
        ]
        # L-R flip：大脑左右对称 → axial/coronal 有效，sagittal 不适合（前后不对称）
        if view in ('axial', 'coronal'):
            aug.append(transforms.RandomHorizontalFlip(p=0.5))
        # U-D flip：仅 axial（海马体层面上下粗略对称），coronal/sagittal 皮层-脑干不对称
        if view == 'axial':
            aug.append(transforms.RandomVerticalFlip(p=0.5))
        aug += [
            # 透视形变：模拟不同 MRI 扫描角度，增加几何多样性
            transforms.RandomPerspective(distortion_scale=0.15, p=0.3),
            # 高斯模糊：模拟不同 MRI 分辨率和采集参数
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.0, hue=0.0),
            transforms.ToTensor(),
            norm,
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.10), ratio=(0.3, 3.3)),
        ]
        return transforms.Compose(aug)
    return transforms.Compose([transforms.Resize(224), transforms.ToTensor(), norm])


class OASISDataset(Dataset):
    def __init__(self, split: str, view: str, slices_dir: str = SLICES_DIR):
        self.transform    = build_transforms(split, view=view)
        self.class_to_idx = {c: i for i, c in enumerate(CLASS_ORDER)}
        self.samples: list[tuple[str, int, str]] = []
        for cls_name in CLASS_ORDER:
            label   = self.class_to_idx[cls_name]
            cls_dir = Path(slices_dir) / view / split / cls_name
            for p in sorted(cls_dir.glob('*.png')):
                sid = '_'.join(p.stem.split('_')[:-1])
                self.samples.append((str(p), label, sid))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, sid = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label, sid

    def class_counts(self) -> dict:
        return dict(Counter(CLASS_ORDER[lbl] for _, lbl, _ in self.samples))


def collate_fn(batch):
    imgs   = torch.stack([b[0] for b in batch])
    labels = torch.tensor([b[1] for b in batch])
    sids   = [b[2] for b in batch]
    return imgs, labels, sids


print('Building tri-view datasets (augmentation v2)...')
t = time.time()

train_datasets, val_datasets, test_datasets = {}, {}, {}
train_loaders,  val_loaders,  test_loaders  = {}, {}, {}

ldr_kw = dict(num_workers=NUM_WORKERS, persistent_workers=True, pin_memory=True,
              collate_fn=collate_fn)

for view in VIEW_CONFIGS:
    train_datasets[view] = OASISDataset('train', view)
    val_datasets[view]   = OASISDataset('val',   view)
    test_datasets[view]  = OASISDataset('test',  view)
    train_loaders[view]  = DataLoader(train_datasets[view], batch_size=BATCH_SIZE,     shuffle=True,  **ldr_kw)
    val_loaders[view]    = DataLoader(val_datasets[view],   batch_size=BATCH_SIZE * 2, shuffle=False, **ldr_kw)
    test_loaders[view]   = DataLoader(test_datasets[view],  batch_size=BATCH_SIZE * 2, shuffle=False, **ldr_kw)

print(f'  Done in {time.time()-t:.1f}s')

nd_cnt = train_datasets['axial'].class_counts().get('NonDemented', 1)
d_cnt  = train_datasets['axial'].class_counts().get('Demented', 1)
total  = nd_cnt + d_cnt
class_weights = torch.tensor(
    [total / (2 * nd_cnt), total / (2 * d_cnt)], dtype=torch.float32, device=DEVICE)

print(f"\n{'View':<12} {'Split':<8} {'NonDemented':>14} {'Demented':>10} {'Total':>8}")
print('-' * 56)
for view in VIEW_CONFIGS:
    for ds, name in [(train_datasets[view], 'train'), (val_datasets[view], 'val'), (test_datasets[view], 'test')]:
        c = ds.class_counts()
        nd_c, d_c = c.get('NonDemented', 0), c.get('Demented', 0)
        print(f'{view:<12} {name:<8} {nd_c:>14} {d_c:>10} {nd_c+d_c:>8}')
print(f'\nClass weights: NonDemented={class_weights[0]:.3f}, Demented={class_weights[1]:.3f}')
print('\n增强策略 v2（train）:')
print('  All views : Resize(256)→RandomCrop(224) + Affine(±15°,±5%,×0.9-1.1)')
print('              + RandomPerspective(0.15,p=0.3) + GaussianBlur(3,σ0.1-1.5)')
print('              + ColorJitter(0.3) + RandomErasing(0.2)')
print('  axial+coronal : + RandomHorizontalFlip(0.5)')
print('  axial only    : + RandomVerticalFlip(0.5)')

In [ ]:
# ── DSNet: DenseNet-121 (CNN) → Feature Projection → Swin-T (Transformer) ──
#
# 与 Kaggle 版架构完全相同，仅 num_classes 改为 2
#
# Data flow:
#   [B,3,224,224]
#   → cnn_extractor (DenseNet-121 up to denseblock3)  [B,1024,14,14]
#   → feature_proj  (BN+Conv1x1+GELU+Permute)         [B,14,14,384]  NHWC
#   → swin_stage3   (6 Swin blocks, pretrained)        [B,14,14,384]
#   → swin_merging  (PatchMerging, pretrained)         [B, 7, 7,768]
#   → swin_stage4   (2 Swin blocks, pretrained) *CAM*  [B, 7, 7,768]
#   → swin_norm     (LayerNorm)                        [B, 7, 7,768]
#   → AvgPool+Flatten                                  [B, 768]
#   → classifier    (Dropout+Linear)                   [B, 2]

class DSNet(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()

        dn = torchvision.models.densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        f  = dn.features
        self.cnn_extractor = nn.Sequential(
            f.conv0, f.norm0, f.relu0, f.pool0,
            f.denseblock1, f.transition1,
            f.denseblock2, f.transition2,
            f.denseblock3,
        )

        self.feature_proj = nn.Sequential(
            nn.BatchNorm2d(1024),
            nn.Conv2d(1024, 384, kernel_size=1, bias=False),
            nn.GELU(),
        )

        st = torchvision.models.swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)
        self.swin_stage3  = st.features[5]
        self.swin_merging = st.features[6]
        self.swin_stage4  = st.features[7]   # Grad-CAM target
        self.swin_norm    = st.norm

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(768, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.cnn_extractor(x)          # [B, 1024, 14, 14]
        x = self.feature_proj(x)           # [B, 384, 14, 14]
        x = x.permute(0, 2, 3, 1)         # [B, 14, 14, 384]  NHWC
        x = self.swin_stage3(x)            # [B, 14, 14, 384]
        x = self.swin_merging(x)           # [B,  7,  7, 768]
        x = self.swin_stage4(x)            # [B,  7,  7, 768]  <- Grad-CAM
        x = self.swin_norm(x)              # [B,  7,  7, 768]
        x = x.permute(0, 3, 1, 2)         # [B, 768, 7, 7]
        x = x.mean(dim=[2, 3])            # [B, 768]
        return self.classifier(x)          # [B, 2]


def freeze_features(model: DSNet) -> None:
    for name, p in model.named_parameters():
        if 'feature_proj' not in name and 'classifier' not in name:
            p.requires_grad = False


def unfreeze_all(model: DSNet) -> None:
    for p in model.parameters():
        p.requires_grad = True


class EarlyStopping:
    def __init__(self, patience: int, min_delta: float, path: str):
        self.patience  = patience
        self.min_delta = min_delta
        self.path      = path
        self.best_acc  = 0.0
        self.counter   = 0

    def step(self, val_acc: float, model: nn.Module) -> bool:
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc = val_acc
            self.counter  = 0
            torch.save({'model_state_dict': model.state_dict(),
                        'val_acc': val_acc}, self.path)
            return False
        self.counter += 1
        return self.counter >= self.patience

    def reset_counter(self):
        self.counter = 0


model = DSNet(num_classes=2).to(DEVICE)
total  = sum(p.numel() for p in model.parameters())
n_proj = sum(p.numel() for p in model.feature_proj.parameters())
n_cls  = sum(p.numel() for p in model.classifier.parameters())
print(f'DSNet (binary) total params : {total:,}')
print(f'Phase 1 trainable (proj+head): {n_proj+n_cls:,}')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch, phase):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f'P{phase}|E{epoch:02d}', leave=False)
    for imgs, labels, _ in pbar:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', enabled=USE_AMP):
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
        total_loss += loss.item() * labels.size(0)
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{correct/total:.4f}')
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels, _ in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with autocast(device_type='cuda', enabled=USE_AMP):
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
        total_loss += loss.item() * labels.size(0)
    return total_loss / total, correct / total


def run_phase(phase, model, criterion, optimizer, scheduler, stopper, n_epochs,
              tr_loader, vl_loader) -> list:
    """训练一个 Phase，返回每 epoch 的 history 列表，供绘制训练曲线使用"""
    scaler  = GradScaler('cuda', enabled=USE_AMP)
    history = []
    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, tr_loader, criterion, optimizer, scaler, epoch, phase)
        vl_loss, vl_acc = validate(model, vl_loader, criterion)
        scheduler.step()
        lr     = optimizer.param_groups[0]['lr']
        marker = ' <- best' if vl_acc >= stopper.best_acc else ''
        print(f'  P{phase}|E{epoch:02d}  '
              f'train={tr_acc:.4f}  val={vl_acc:.4f}  loss={vl_loss:.4f}  '
              f'lr={lr:.2e}{marker}')
        history.append({
            'phase': phase, 'epoch': epoch,
            'tr_acc': tr_acc, 'vl_acc': vl_acc,
            'tr_loss': tr_loss, 'vl_loss': vl_loss,
        })
        if stopper.step(vl_acc, model):
            print(f'  Early stopping. Best val acc: {stopper.best_acc:.4f}')
            break
    return history

print('Training functions ready.')

In [ ]:
# ── Evaluation functions（受试者级 + 三视角融合 + 训练曲线）────────

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_labels, all_probs, all_sids = [], [], []
    for imgs, labels, sids in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(device_type='cuda', enabled=USE_AMP):
            outputs = model(imgs)
        probs = torch.softmax(outputs, 1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_probs.append(probs)
        all_sids.extend(sids)
    return np.array(all_labels), np.vstack(all_probs), all_sids


def subject_level_aggregate(labels_arr, probs_arr, sids):
    sid_probs  = defaultdict(list)
    sid_labels = {}
    for i, sid in enumerate(sids):
        sid_probs[sid].append(probs_arr[i])
        sid_labels[sid] = labels_arr[i]
    subj_ids    = list(sid_probs.keys())
    subj_probs  = np.array([np.mean(sid_probs[s], axis=0) for s in subj_ids])
    subj_labels = np.array([sid_labels[s] for s in subj_ids])
    return subj_labels, subj_probs, subj_ids


def find_optimal_threshold(subj_labels, subj_probs_pos,
                           metric: str = 'youden',
                           min_thresh: float = 0.05) -> float:
    """在验证集上寻找最优决策阈值，带最低阈值保护。

    min_thresh=0.05 防止年龄混杂（年轻健康对照 CDR=NaN → label=0）
    导致阈值退化为接近零的极端值（如 0.009）。
    """
    from sklearn.metrics import roc_curve, f1_score as _f1
    fpr, tpr, thresholds = roc_curve(subj_labels, subj_probs_pos)
    valid_mask = thresholds >= min_thresh
    if valid_mask.sum() < 2:
        valid_mask = np.ones(len(thresholds), dtype=bool)
    thresholds_v = thresholds[valid_mask]
    fpr_v        = fpr[valid_mask]
    tpr_v        = tpr[valid_mask]
    if metric == 'youden':
        best_idx = int(np.argmax(tpr_v - fpr_v))
    else:
        f1s = [_f1(subj_labels, (subj_probs_pos >= t).astype(int),
                   pos_label=1, zero_division=0) for t in thresholds_v]
        best_idx = int(np.argmax(f1s))
    return float(thresholds_v[best_idx])


def compute_metrics(subj_labels, subj_probs, threshold: float = 0.5):
    from sklearn.metrics import f1_score
    pos_probs = subj_probs[:, 1]
    preds     = (pos_probs >= threshold).astype(int)
    auc       = roc_auc_score(subj_labels, pos_probs)
    cm        = confusion_matrix(subj_labels, preds)
    tn, fp, fn, tp = cm.ravel()
    f1_demented        = f1_score(subj_labels, preds, pos_label=1, zero_division=0)
    precision_demented = tp / (tp + fp + 1e-9)
    return {
        'auc_roc':     auc,
        'accuracy':    (tp + tn) / (tp + tn + fp + fn),
        'sensitivity': tp / (tp + fn + 1e-9),
        'specificity': tn / (tn + fp + 1e-9),
        'precision':   precision_demented,
        'f1_demented': f1_demented,
        'threshold':   threshold,
        'cm':     cm,
        'report': classification_report(subj_labels, preds, target_names=CLASS_ORDER, digits=4),
        'labels': subj_labels,
        'probs':  pos_probs,
        'n':      len(subj_labels),
    }


def evaluate(model, loader, threshold: float = 0.5):
    labels, probs, sids = collect_predictions(model, loader)
    subj_labels, subj_probs, _ = subject_level_aggregate(labels, probs, sids)
    return compute_metrics(subj_labels, subj_probs, threshold=threshold)


def evaluate_fusion(models_dict, loaders_dict, views=None, threshold: float = 0.5,
                    weights: dict = None):
    """受试者级多视角融合评估。

    weights: 可选的视角权重字典（如 {'axial': 0.82, 'coronal': 0.79, 'sagittal': 0.81}）。
             None 时使用等权平均；提供时按权重加权平均概率向量。
    """
    if views is None:
        views = list(models_dict.keys())
    sid_probs   = defaultdict(list)
    sid_weights = defaultdict(list)
    sid_labels  = {}
    for view in views:
        w = weights[view] if weights is not None else 1.0
        labels, probs, sids = collect_predictions(models_dict[view], loaders_dict[view])
        for i, sid in enumerate(sids):
            sid_probs[sid].append(probs[i] * w)
            sid_weights[sid].append(w)
            sid_labels[sid] = labels[i]
    subj_ids    = list(sid_probs.keys())
    subj_probs  = np.array([
        np.sum(sid_probs[s], axis=0) / np.sum(sid_weights[s])
        for s in subj_ids
    ])
    subj_labels = np.array([sid_labels[s] for s in subj_ids])
    return compute_metrics(subj_labels, subj_probs, threshold=threshold)


# ── 训练曲线可视化 ────────────────────────────────────────────────

def plot_training_curves(history: list, view: str, best_epoch: int, path: str):
    epochs   = [h['epoch_global'] for h in history]
    tr_accs  = [h['tr_acc']  for h in history]
    vl_accs  = [h['vl_acc']  for h in history]
    tr_losses= [h['tr_loss'] for h in history]
    vl_losses= [h['vl_loss'] for h in history]
    p1_epochs = [h for h in history if h['phase'] == 1]
    p1_end    = len(p1_epochs)
    has_p2    = any(h['phase'] == 2 for h in history)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Training Curves — DSNet ({view}) | OASIS-1', fontsize=12, fontweight='bold')

    for ax, tr_vals, vl_vals, ylabel, title in [
        (ax1, tr_accs,  vl_accs,  'Accuracy', 'Accuracy'),
        (ax2, tr_losses,vl_losses,'Loss',      'Loss'),
    ]:
        ax.plot(epochs, tr_vals, 'b-o', markersize=3, label='Train')
        ax.plot(epochs, vl_vals, 'r-o', markersize=3, label='Val')
        ax.axvspan(0.5, p1_end + 0.5, alpha=0.07, color='blue', label='Phase 1 (frozen)')
        if has_p2:
            ax.axvspan(p1_end + 0.5, max(epochs) + 0.5, alpha=0.07, color='orange',
                       label='Phase 2 (finetune)')
        ax.axvline(x=best_epoch, color='green', linestyle='--', linewidth=1.5,
                   label=f'Best ckpt (E{best_epoch})')
        ax.set_xlabel('Global Epoch'); ax.set_ylabel(ylabel); ax.set_title(title)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {path}')


# ── 过拟合分析 ────────────────────────────────────────────────────

def analyze_overfitting(history: list, view: str):
    best_vl_acc = max(h['vl_acc'] for h in history)
    best_ep     = next(h['epoch_global'] for h in history if h['vl_acc'] == best_vl_acc)
    best_tr_acc = next(h['tr_acc'] for h in history if h['epoch_global'] == best_ep)
    gap         = best_tr_acc - best_vl_acc

    vl_losses  = [h['vl_loss'] for h in history]
    min_vl_idx = int(np.argmin(vl_losses))
    post_min   = vl_losses[min_vl_idx:]
    diverged   = len(post_min) > 2 and post_min[-1] > post_min[0] * 1.05

    severity = '✅ 正常' if gap < 0.05 else ('⚠️  轻微过拟合' if gap < 0.12 else '❌ 中度过拟合')
    print(f'\n[过拟合分析] {view}')
    print(f'  Best epoch      : E{best_ep}')
    print(f'  Train acc       : {best_tr_acc:.4f}')
    print(f'  Val acc (best)  : {best_vl_acc:.4f}')
    print(f'  Gap (train-val) : {gap:.4f}  →  {severity}')
    print(f'  Val loss diverge: {"是" if diverged else "否"}')
    return best_ep, gap


# ── 消融表格（含 Precision 和 Threshold 列）──────────────────────

def print_ablation_table(results: dict):
    header = (f"{'方法':<28} {'N':>5} {'AUC-ROC':>9} {'Acc':>7} "
              f"{'Sens':>7} {'Spec':>7} {'Prec':>7} {'F1(D)':>7} {'Thresh':>8}")
    print(header)
    print('-' * len(header))
    for name, r in results.items():
        print(f"{name:<28} {r['n']:>5} {r['auc_roc']:>9.4f} {r['accuracy']:>7.4f} "
              f"{r['sensitivity']:>7.4f} {r['specificity']:>7.4f} {r.get('precision', float('nan')):>7.4f} "
              f"{r['f1_demented']:>7.4f} {r.get('threshold', 0.5):>8.3f}")

    print('\n--- LaTeX 格式 ---')
    print(r'\begin{tabular}{lrrrrrrrr}')
    print(r'\hline')
    print(r'Method & N & AUC-ROC & Accuracy & Sensitivity & Specificity & Precision & F1 (Demented) & Threshold \\')
    print(r'\hline')
    for name, r in results.items():
        print(f"{name} & {r['n']} & {r['auc_roc']:.4f} & {r['accuracy']:.4f} "
              f"& {r['sensitivity']:.4f} & {r['specificity']:.4f} "
              f"& {r.get('precision', float('nan')):.4f} "
              f"& {r['f1_demented']:.4f} & {r.get('threshold', 0.5):.3f} \\\\")
    print(r'\hline')
    print(r'\end{tabular}')


def plot_confusion_matrix(cm, title, path):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(CLASS_ORDER, rotation=20, ha='right')
    ax.set_yticklabels(CLASS_ORDER)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    th = cm.max() / 2
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > th else 'black', fontsize=13)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def plot_roc_comparison(results: dict, path):
    fig, ax = plt.subplots(figsize=(7, 6))
    for name, r in results.items():
        fpr, tpr, _ = roc_curve(r['labels'], r['probs'])
        ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={r["auc_roc"]:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves — DSNet Tri-View Ablation (OASIS-1)')
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()
    print(f'  Saved: {path}')

print('Evaluation functions ready.')

In [ ]:
# ── Grad-CAM++ for DSNet (target layer: swin_stage4) ──────────────
# 使用 GradCAMPlusPlus（比原版 GradCAM 定位更锐利）
# 生成时对热力图应用脑掩膜，防止激活"漏"到黑色背景外

BIOMARKER = {
    'NonDemented': 'Normal hippocampus | intact cortical thickness | no focal atrophy',
    'Demented':    'Hippocampal atrophy | medial temporal lobe | ventricular enlargement',
}


def reshape_transform(tensor: torch.Tensor) -> torch.Tensor:
    return tensor.permute(0, 3, 1, 2).contiguous()


def setup_gradcam(model: DSNet) -> GradCAMPlusPlus:
    """使用 GradCAMPlusPlus（二阶梯度加权，比 GradCAM 定位更锐利）"""
    return GradCAMPlusPlus(
        model=model,
        target_layers=[model.swin_stage4],
        reshape_transform=reshape_transform,
    )


@torch.no_grad()
def select_samples(model, dataset, n: int = 5) -> dict:
    model.eval()
    hits = {i: [] for i in range(2)}
    for idx in tqdm(range(len(dataset)), desc='Selecting Grad-CAM samples', leave=False):
        img, label, _ = dataset[idx]
        out  = model(img.unsqueeze(0).to(DEVICE))
        prob = torch.softmax(out, 1)
        pred = out.argmax(1).item()
        if pred == label:
            hits[label].append((idx, prob[0, pred].item()))
    return {c: [x[0] for x in sorted(v, key=lambda x: x[1], reverse=True)[:n]]
            for c, v in hits.items()}


def generate_gradcam(model, cam, dataset, selected: dict, view: str = '') -> None:
    """生成 Grad-CAM++ 热力图，应用脑掩膜过滤背景激活。
    
    背景像素（黑色，值≈0）经 ImageNet 归一化后变为约 -2.1，
    导致边界梯度极大，热力图"漏"到脑外。
    brain_mask 将脑区外的热力值强制为 0，确保激活只在脑内显示。
    """
    model.eval()
    prefix = f'{view}_' if view else ''
    for cls_idx, cls_name in enumerate(CLASS_ORDER):
        idxs = selected.get(cls_idx, [])
        if not idxs:
            print(f'  No samples for {cls_name}'); continue
        n   = len(idxs)
        fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
        if n == 1:
            axes = np.array([[axes[0]], [axes[1]]])
        fig.suptitle(
            f'Grad-CAM++ [DSNet OASIS-1] ({view}): {cls_name}\n'
            f'Expected activation: {BIOMARKER[cls_name]}',
            fontsize=10,
        )
        for col, si in enumerate(idxs):
            path, _, _ = dataset.samples[si]
            raw_img  = np.array(
                Image.open(path).convert('RGB').resize((224, 224)),
                dtype=np.float32,
            ) / 255.0
            norm_t, _, _ = dataset[si]
            gs_cam = cam(
                input_tensor=norm_t.unsqueeze(0).to(DEVICE),
                targets=[ClassifierOutputTarget(cls_idx)],
            )[0]
            # 脑掩膜：原始图像亮度 > 2% 视为脑区（背景为纯黑 = 0）
            brain_mask    = (raw_img.mean(axis=2) > 0.02).astype(np.float32)
            gs_cam_masked = gs_cam * brain_mask   # 背景热力值归零
            overlay = show_cam_on_image(
                raw_img, gs_cam_masked,
                use_rgb=True, colormap=cv2.COLORMAP_JET, image_weight=0.5,
            )
            axes[0, col].imshow(raw_img, cmap='gray')
            axes[0, col].set_title('Original', fontsize=8)
            axes[0, col].axis('off')
            axes[1, col].imshow(overlay)
            axes[1, col].set_title('Grad-CAM++', fontsize=8)
            axes[1, col].axis('off')
        plt.tight_layout()
        save_p = f'{GRADCAM_DIR}/gradcam_{prefix}{cls_name}.png'
        plt.savefig(save_p, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'  Saved: {save_p}')

print('Grad-CAM++ functions ready (with brain mask).')

In [ ]:
# ── Run Full Pipeline: 三视角独立训练（Phase 1 + Phase 2 视角专用LR）+ 消融评估 ──
# sagittal Phase 2 LR_PROJ = 5e-5（原 1e-5 过低，导致检查点从未更新，AUC 骤降至 0.716）

from IPython.display import Image as IPImage, display

t0        = time.time()
criterion = nn.CrossEntropyLoss(weight=class_weights)

trained_models  = {}
all_histories   = {}
best_thresholds = {}

# ── 循环训练三个视角 ──────────────────────────────────────────────
for view in VIEW_CONFIGS:
    print(f'\n{"="*60}')
    print(f'[{view.upper()}]  训练 DSNet_{view}')
    print(f'{"="*60}')

    model     = DSNet(num_classes=2).to(DEVICE)
    ckpt_path = f'{CHECKPOINT_DIR}/dsnet_{view}_best.pth'
    stopper   = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA, path=ckpt_path)

    # Phase 1
    print(f'[Phase 1] 冻结 CNN+Swin，训练 proj+head')
    freeze_features(model)
    print(f'  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}\n')

    opt1 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS, eta_min=1e-6)
    hist1 = run_phase(1, model, criterion, opt1, sch1, stopper, PHASE1_EPOCHS,
                      train_loaders[view], val_loaders[view])
    for h in hist1:
        h['epoch_global'] = h['epoch']

    # Phase 2（视角专用 proj LR）
    proj_lr = VIEW_PHASE2_LR_PROJ[view]
    print(f'\n[Phase 2] 全层微调（proj LR={proj_lr:.0e}，视角专用）')
    ckpt = torch.load(ckpt_path)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'  Loaded Phase 1 best (val_acc={ckpt["val_acc"]:.4f})\n')

    unfreeze_all(model)
    stopper.reset_counter()

    cnn_p  = list(model.cnn_extractor.parameters())
    swin_p = (list(model.swin_stage3.parameters()) + list(model.swin_merging.parameters()) +
              list(model.swin_stage4.parameters()) + list(model.swin_norm.parameters()))
    proj_p = list(model.feature_proj.parameters()) + list(model.classifier.parameters())

    opt2 = torch.optim.AdamW([
        {'params': cnn_p,  'lr': PHASE2_LR_CNN},
        {'params': swin_p, 'lr': PHASE2_LR_SWIN},
        {'params': proj_p, 'lr': proj_lr},
    ], weight_decay=WEIGHT_DECAY)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS, eta_min=1e-8)
    hist2 = run_phase(2, model, criterion, opt2, sch2, stopper, PHASE2_EPOCHS,
                      train_loaders[view], val_loaders[view])
    p1_len = len(hist1)
    for h in hist2:
        h['epoch_global'] = p1_len + h['epoch']

    full_history = hist1 + hist2
    all_histories[view] = full_history

    ckpt = torch.load(ckpt_path)
    model.load_state_dict(ckpt['model_state_dict'])
    trained_models[view] = model
    print(f'  [{view}] 训练完成，best val_acc={ckpt["val_acc"]:.4f}')

    # ── 在 val 集上寻找最优阈值（Youden's J，min_thresh=0.05）──
    val_labels_, val_probs_, val_sids_ = collect_predictions(model, val_loaders[view])
    val_subj_labels, val_subj_probs, _ = subject_level_aggregate(val_labels_, val_probs_, val_sids_)
    opt_thresh = find_optimal_threshold(val_subj_labels, val_subj_probs[:, 1], metric='youden')
    best_thresholds[view] = opt_thresh
    print(f'  [{view}] 最优阈值（Youden，val set）: {opt_thresh:.3f}')

    best_ep, gap = analyze_overfitting(full_history, view)

    curve_path = f'{RESULTS_DIR}/training_curves_{view}.png'
    plot_training_curves(full_history, view, best_ep, curve_path)
    display(IPImage(curve_path))

print(f'\n最优阈值汇总: {best_thresholds}')

# ── 消融实验评估 ───────────────────────────────────────────────────
print(f'\n{"="*60}')
print('[ABLATION]  消融实验（受试者级，Test Set，最优阈值）')
print(f'{"="*60}\n')

ablation_results = {}

for view, model in trained_models.items():
    ablation_results[f'DSNet-{view}'] = evaluate(
        model, test_loaders[view], threshold=best_thresholds[view])

for v1, v2 in [('axial', 'coronal'), ('axial', 'sagittal'), ('coronal', 'sagittal')]:
    thresh = float(np.mean([best_thresholds[v1], best_thresholds[v2]]))
    ablation_results[f'DSNet-{v1}+{v2}'] = evaluate_fusion(
        trained_models, test_loaders, views=[v1, v2], threshold=thresh)

thresh_tri = float(np.mean(list(best_thresholds.values())))
ablation_results['DSNet-TriView (ours)'] = evaluate_fusion(
    trained_models, test_loaders, threshold=thresh_tri)

# ── AUC 加权融合（val AUC 作为权重，论文附加行）─────────────────
val_aucs = {}
for view, model in trained_models.items():
    r_val = evaluate(model, val_loaders[view])
    val_aucs[view] = r_val['auc_roc']
print(f'\nVal AUC weights: { {v: f"{w:.4f}" for v, w in val_aucs.items()} }')

ablation_results['DSNet-TriView-weighted'] = evaluate_fusion(
    trained_models, test_loaders, weights=val_aucs, threshold=thresh_tri)

print_ablation_table(ablation_results)

# ── 固定阈值 0.5 对照组 ────────────────────────────────────────────
print(f'\n{"="*60}')
print('[对照]  固定阈值 0.5')
print(f'{"="*60}\n')

ablation_fixed = {}
for view, model in trained_models.items():
    ablation_fixed[f'DSNet-{view} (t=0.5)'] = evaluate(model, test_loaders[view], threshold=0.5)
ablation_fixed['DSNet-TriView (t=0.5)'] = evaluate_fusion(trained_models, test_loaders, threshold=0.5)
ablation_fixed['DSNet-TriView-w (t=0.5)'] = evaluate_fusion(
    trained_models, test_loaders, weights=val_aucs, threshold=0.5)
print_ablation_table(ablation_fixed)

# ── 可视化 ────────────────────────────────────────────────────────
roc_path = f'{RESULTS_DIR}/roc_triview_ablation.png'
plot_roc_comparison(ablation_results, roc_path)
display(IPImage(roc_path))

tri_r   = ablation_results['DSNet-TriView (ours)']
cm_path = f'{RESULTS_DIR}/confusion_matrix_triview.png'
plot_confusion_matrix(
    tri_r['cm'],
    f'Confusion Matrix — Tri-View Fusion (Subject-Level, thresh={thresh_tri:.3f})',
    cm_path)
display(IPImage(cm_path))

# ── 保存报告 ──────────────────────────────────────────────────────
rp = f'{RESULTS_DIR}/ablation_report.txt'
with open(rp, 'w') as f:
    f.write('=== DSNet Tri-View Ablation Study — OASIS-1 ===\n\n')
    f.write(f'Optimal thresholds (Youden, val set): {best_thresholds}\n')
    f.write(f'Val AUC weights: {val_aucs}\n\n')
    f.write(f"{'Method':<28} {'N':>5} {'AUC':>8} {'Acc':>7} {'Sens':>7} "
            f"{'Spec':>7} {'Prec':>7} {'F1(D)':>7} {'Thresh':>8}\n")
    f.write('-' * 80 + '\n')
    for name, r in ablation_results.items():
        f.write(f'{name:<28} {r["n"]:>5} {r["auc_roc"]:>8.4f} {r["accuracy"]:>7.4f} '
                f'{r["sensitivity"]:>7.4f} {r["specificity"]:>7.4f} '
                f'{r.get("precision", float("nan")):>7.4f} '
                f'{r["f1_demented"]:>7.4f} {r.get("threshold", 0.5):>8.3f}\n')
    f.write('\n=== Tri-View Fusion Classification Report ===\n')
    f.write(tri_r['report'])
    f.write('\n=== Overfitting Summary ===\n')
    for view, hist in all_histories.items():
        best_ep, gap = analyze_overfitting(hist, view)
        f.write(f'{view}: train-val gap={gap:.4f}\n')
print(f'Report saved: {rp}')

elapsed = (time.time() - t0) / 60
print(f'\n{"="*60}')
print('TRI-VIEW DSNet COMPLETE')
print(f'{"="*60}')
print(f'  Tri-View AUC-ROC   : {tri_r["auc_roc"]:.4f}')
print(f'  Tri-View Accuracy  : {tri_r["accuracy"]:.4f}')
print(f'  Tri-View Precision : {tri_r.get("precision", float("nan")):.4f}')
print(f'  Tri-View F1(Dement): {tri_r["f1_demented"]:.4f}')
print(f'  Tri-View Threshold : {thresh_tri:.3f}')
print(f'  Val AUC weights    : {val_aucs}')
print(f'  Total time         : {elapsed:.1f} min')
print(f'  Checkpoints        : {CHECKPOINT_DIR}')

In [ ]:
# ── Grad-CAM++ 三视角可视化（pipeline 训练完成后运行）────────────
# 对每个视角的最优模型，在测试集上选取正确分类的样本，生成热力图
# 输出：gradcam/gradcam_{view}_{cls_name}.png  共 3×2=6 张

from IPython.display import Image as IPImage, display

print('=== Grad-CAM++ 三视角可视化 ===')

for view, model in trained_models.items():
    print(f'\n[Grad-CAM++] {view} 视角（测试集，置信度最高的正确分类样本）')
    cam      = setup_gradcam(model)
    selected = select_samples(model, test_datasets[view], n=N_GRADCAM)
    generate_gradcam(model, cam, test_datasets[view], selected, view=view)
    del cam   # 析构时自动移除 forward/backward hook

    for cls_name in CLASS_ORDER:
        img_path = f'{GRADCAM_DIR}/gradcam_{view}_{cls_name}.png'
        if os.path.exists(img_path):
            print(f'  [{view}] {cls_name}:')
            display(IPImage(img_path))

print(f'\nGrad-CAM++ 完成。共生成 {len(trained_models) * 2} 张图（{len(trained_models)} 视角 × 2 类别）')
print(f'保存目录: {GRADCAM_DIR}')